# 🎓 Smart Classroom Occupancy — Improved Training (v2)
### What's new in v2:
- **Strong data augmentation** → handles sitting, walking, far, near, talking
- **Balanced sampling** → equal frames from all classes
- **Mixed images + videos** → you can add manual photos
- **Better validation** → separate test set for real accuracy

In [ ]:
# Step 1: Install dependencies
!pip install torch torchvision --quiet
!pip install gdown --quiet
print("✅ All packages ready")

In [ ]:
# Step 2: Connect Google Drive (where your dataset folder is)
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive connected")

In [ ]:
# Step 3: Configuration — update DATASET_DIR to your folder path
import os

# ── CHANGE THIS to your Drive folder path ──────────────────────────────
DATASET_DIR = '/content/drive/MyDrive/classroom_dataset_v2'
# ────────────────────────────────────────────────────────────────────────

# Verify structure
for cls in ['low', 'medium', 'high']:
    path = os.path.join(DATASET_DIR, cls)
    if os.path.exists(path):
        n = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"  {cls:10s}: {n} images")
    else:
        print(f"  ❌ Missing folder: {path}")

IMAGE_SIZE  = 224
BATCH_SIZE  = 16    # lower if you get out-of-memory errors
EPOCHS      = 40
LR          = 1e-3

In [ ]:
# Step 4 (OPTIONAL): Extract frames from existing videos automatically
# Run this if you want to pull frames from video files in your Drive
# Skip this cell if you already have images in the dataset folders

import cv2, os

def extract_frames(video_path, output_folder, every_n_seconds=2):
    os.makedirs(output_folder, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    interval = int(fps * every_n_seconds)
    count = saved = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % interval == 0:
            out = os.path.join(output_folder, f"frame_{count:06d}.jpg")
            cv2.imwrite(out, frame)
            saved += 1
        count += 1
    cap.release()
    print(f"  Extracted {saved} frames from {os.path.basename(video_path)}")

# ── SET VIDEO PATHS HERE (comment out lines you don't need) ─────────────
# extract_frames('/content/drive/MyDrive/videos/low_video.mp4',
#                os.path.join(DATASET_DIR, 'low'),
#                every_n_seconds=1)   # 1 frame per second

# extract_frames('/content/drive/MyDrive/videos/medium_video.mp4',
#                os.path.join(DATASET_DIR, 'medium'),
#                every_n_seconds=1)

# extract_frames('/content/drive/MyDrive/videos/high_video.mp4',
#                os.path.join(DATASET_DIR, 'high'),
#                every_n_seconds=1)

print("Frame extraction done (or skipped)")

In [ ]:
# Step 5: Strong data augmentation — handles all poses and distances
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np

# ── Transforms ────────────────────────────────────────────────────────────
# Training: heavy augmentation so model learns sitting, walking, far, near
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0)),  # zoom in/out
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.4,   # lighting changes
        contrast=0.4,
        saturation=0.3,
        hue=0.1
    ),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),  # blur = people far away
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),  # occlusion simulation
])

# Validation: no augmentation, just resize + normalize
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Load dataset ──────────────────────────────────────────────────────────
full_dataset = datasets.ImageFolder(DATASET_DIR, transform=train_transform)
CLASS_NAMES  = full_dataset.classes
print(f"Classes (alphabetical): {CLASS_NAMES}")
print(f"Total images: {len(full_dataset)}")

# ── Train / Val split (80/20) ─────────────────────────────────────────────
n       = len(full_dataset)
n_val   = max(int(n * 0.2), len(CLASS_NAMES))
n_train = n - n_val

train_ds, val_ds = torch.utils.data.random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)
# Apply val transform to val set
val_ds.dataset = datasets.ImageFolder(DATASET_DIR, transform=val_transform)

# ── Balanced sampler (handles unequal class sizes) ────────────────────────
labels      = [full_dataset.targets[i] for i in train_ds.indices]
class_count = np.bincount(labels)
weights     = 1.0 / class_count[labels]
sampler     = WeightedRandomSampler(weights, num_samples=len(labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,     num_workers=2)

print(f"Train: {n_train} | Val: {n_val}")
print(f"Class counts: { {c: int(class_count[i]) for i,c in enumerate(CLASS_NAMES)} }")

In [ ]:
# Step 6: CNN model (same architecture as v1 — compatible with existing app)
import torch.nn as nn

class ClassroomCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.1),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.1),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.2),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256),
            nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.2),
        )
        self.gap   = nn.AdaptiveAvgPool2d((1, 1))
        self.brain = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.brain(self.gap(self.features(x)))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = ClassroomCNN(num_classes=len(CLASS_NAMES)).to(device)
print(f"Model ready on: {device}")
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,}")

In [ ]:
# Step 7: Train the model
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_val_acc = 0.0
history      = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────
    model.train()
    t_loss = t_correct = t_total = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss    += loss.item()
        t_correct += (out.argmax(1) == labels).sum().item()
        t_total   += labels.size(0)
    scheduler.step()

    # ── Validate ───────────────────────────────────────────────────────
    model.eval()
    v_loss = v_correct = v_total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out   = model(imgs)
            loss  = criterion(out, labels)
            v_loss    += loss.item()
            v_correct += (out.argmax(1) == labels).sum().item()
            v_total   += labels.size(0)

    t_acc = 100 * t_correct / t_total
    v_acc = 100 * v_correct / v_total
    history["train_loss"].append(t_loss / len(train_loader))
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss / len(val_loader))
    history["val_acc"].append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save({"model_state_dict": model.state_dict(),
                    "class_names": CLASS_NAMES,
                    "image_size": IMAGE_SIZE,
                    "epoch": epoch,
                    "val_acc": v_acc},
                   "classroom_occupancy_model_v2.pth")
        saved = " ← BEST saved"
    else:
        saved = ""

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | "
              f"Train {t_acc:.1f}% | Val {v_acc:.1f}%{saved}")

print(f"\n✅ Training complete! Best val accuracy: {best_val_acc:.1f}%")

In [ ]:
# Step 8: Plot training results
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history['train_acc']) + 1)

ax1.plot(epochs, history['train_acc'], label='Train', color='#00d4ff')
ax1.plot(epochs, history['val_acc'],   label='Val',   color='#ffb300')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('%')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history['train_loss'], label='Train', color='#00d4ff')
ax2.plot(epochs, history['val_loss'],   label='Val',   color='#ffb300')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curve_v2.png', dpi=120)
plt.show()
print("Saved: training_curve_v2.png")

In [ ]:
# Step 9: Per-class accuracy breakdown
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("\n=== Per-Class Accuracy ===")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(all_labels, all_preds)
print("Rows=Actual, Cols=Predicted")
header = ''.join(f'{c:>10}' for c in CLASS_NAMES)
print(f"{'':>10}{header}")
for i, row in enumerate(cm):
    print(f"{CLASS_NAMES[i]:>10}" + ''.join(f'{v:>10}' for v in row))

In [ ]:
# Step 10: Save class config and copy everything to your Drive
import json, shutil

config = {"class_names": CLASS_NAMES, "image_size": IMAGE_SIZE}
with open("class_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Saved class_config.json:", config)

# Copy to Drive
output_dir = '/content/drive/MyDrive/classroom_model_v2'
os.makedirs(output_dir, exist_ok=True)

for fname in ['classroom_occupancy_model_v2.pth', 'class_config.json',
              'training_curve_v2.png']:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(output_dir, fname))
        print(f"✅ Copied {fname} → {output_dir}")

print("\n🎉 Done! Download these files:")
print("  1. classroom_occupancy_model_v2.pth")
print("  2. class_config.json")
print("Then rename .pth to classroom_occupancy_model.pth and copy to edge_app/model/")